In [1]:
!pip install -q -U transformers datasets accelerate peft trl "torchao>=0.16.0"

In [2]:
import gc
import torch

from datasets import load_dataset
from transformers import AutoTokenizer
from trl import KTOConfig, KTOTrainer
from peft import LoraConfig

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB


In [3]:
DATASET_NAME = "argilla/ultrafeedback-binarized-preferences-cleaned-kto"

dataset = load_dataset(DATASET_NAME, split="train")

print(dataset)
print(dataset.column_names)

print("\nFirst example:")
print(dataset[0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Dataset({
    features: ['prompt', 'completion', 'label', 'model', 'average_rating', 'annotations', 'source'],
    num_rows: 230720
})
['prompt', 'completion', 'label', 'model', 'average_rating', 'annotations', 'source']

First example:
{'prompt': 'Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}', 'completion': 'int main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    if (endsWith(country, "Mediterranean")) {\n        cout << "Yes, the country " << cou

In [4]:
# Keep the experiment manageable.
N_TRAIN = 4000
N_EVAL = 100

# Shuffle first so we don't accidentally select a biased slice.
dataset = dataset.shuffle(seed=42)

train_dataset = dataset.select(range(N_TRAIN))
eval_dataset = dataset.select(range(N_TRAIN, N_TRAIN + N_EVAL))

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

print("\nTrain label distribution:")
print(train_dataset["label"].count(True), "desirable")
print(train_dataset["label"].count(False), "undesirable")

Train: 4000
Eval : 100

Train label distribution:
3360 desirable
640 undesirable


In [5]:
for i in range(3):
    sample = train_dataset[i]

    print("=" * 80)
    print("PROMPT:")
    print(sample["prompt"][:500])

    print("\nCOMPLETION:")
    print(sample["completion"][:800])

    print("\nLABEL:", sample["label"])

PROMPT:
Given the task definition and input, reply with output. In this task, you are given a short story consisting of exactly 5 sentences where the second sentence is missing. You are given a candidate for the second sentence and you need to identify if the given sentence connects the first sentence with the rest of the story. Indicate your answer by "Yes" if it connects, otherwise "No". Do not generate anything else apart from "Yes" or "No". The given sentence is incorrect if it changes the subsequen

COMPLETION:
Ye

LABEL: True
PROMPT:
Here's a challenge for you: Can you create a program that not only extracts and visualizes data from a government aid database using Python, but also uses that data to predict future trends in the number of people relying on government aid during a pandemic? Your program should require users to input specific parameters, such as demographic information and location, and then use that information to make predictions. The program should also be able to

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Pad token: <|endoftext|>
EOS token: <|im_end|>


In [7]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print(peft_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'down_proj', 'up_proj', 'k_proj', 'v_proj', 'o_proj', 'q_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [8]:
training_args = KTOConfig(
    output_dir="./qwen-1.5b-kto",

    # Precision
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),

    # Sequence length
    max_length=512,

    # Memory
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    # Training
    num_train_epochs=1,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=50,

    # KTO
    beta=0.1,
    desirable_weight=1.0,
    undesirable_weight=1.0,

    # Evaluation / logging
    logging_strategy="steps",
    logging_steps=10,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # Stability
    max_grad_norm=1.0,

    seed=42,
    report_to="none",

    # Let KTO handle reference log-probs normally
    precompute_ref_log_probs=False,
)

print(training_args)

KTOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.1,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
desirable_weight=1.0,
disable_dropout=True,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpo

In [9]:
trainer = KTOTrainer(
    model=MODEL_NAME,
    ref_model=None,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer,

    peft_config=peft_config,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Extracting KL eval dataset:   0%|          | 0/93 [00:00<?, ? examples/s]

In [10]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available()
               and torch.cuda.is_bf16_supported()
               else torch.float16,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

try:
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    print("✅ LoRA wrapping works")
except Exception as e:
    import traceback
    traceback.print_exc()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
✅ LoRA wrapping works


In [11]:
trainer.model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

baseline = trainer.evaluate()

print("\nBASELINE")
for k, v in baseline.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

In [ ]:
train_result = trainer.train()

print("\nTraining complete.")
print(train_result)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

after = trainer.evaluate()

print("\nAFTER KTO")
for k, v in after.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

In [ ]:
SAVE_PATH = "./qwen-1.5b-kto-final"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Saved to:", SAVE_PATH)

In [ ]:
import torch

device = trainer.model.device

def generate_response(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )


prompts = [
    "Explain why the sky appears blue.",
    "Write a Python function that checks whether a number is prime.",
    "What is the difference between supervised and reinforcement learning?"
]

for prompt in prompts:
    print("=" * 80)
    print("PROMPT:", prompt)
    print("\nRESPONSE:")
    print(generate_response(prompt))